# Analyse temporelle des consommations électriques — ResStock

## Objectif du notebook

Ce notebook constitue une première étape vers la modélisation temporelle de la consommation énergétique résidentielle à partir des données **ResStock**.

L'objectif est d'extraire et préparer un sous-ensemble cohérent de logements utilisant uniquement l'énergie électrique afin d'étudier leurs profils de consommation temporels et préparer l'application de modèles de type **RNN (Recurrent Neural Network)**.

Contrairement au modèle LightGBM précédent, qui prédisait une consommation annuelle à partir de caractéristiques statiques du bâtiment, cette approche exploite la dimension temporelle de la consommation afin de capturer :

- les variations journalières et saisonnières ;
- les habitudes de consommation des occupants ;
- les cycles de fonctionnement des équipements ;
- les phénomènes d'inertie liés au bâtiment.

---

## Données utilisées

Les données ResStock sont organisées en deux grandes catégories :

### 1. Données statiques du bâtiment

Issues de `X.parquet` et `metadata_clean.parquet`, elles contiennent :

- caractéristiques géométriques ;
- propriétés de l'enveloppe ;
- systèmes HVAC ;
- équipements ;
- informations climatiques ;
- caractéristiques des occupants.

Ces variables pourront être réutilisées dans une architecture hybride :

- une branche **MLP** pour les caractéristiques statiques ;
- une branche **RNN** pour les séries temporelles.

Les agrégats physiques développés précédemment seront également conservés :

- **UA** : coefficient global de transmission thermique ;
- **H_ve** : pertes par renouvellement d'air ;
- **C** : capacité thermique du bâtiment ;
- **A_solaire** : surface solaire équivalente ;
- **compacité** : ratio surface déperditive / volume.

---

### 2. Données temporelles de consommation

Les séries temporelles ResStock contiennent l'évolution de la consommation énergétique au cours du temps pour chaque logement.
Ces données seront utilisées comme entrée principale d'un modèle RNN/LSTM/GRU/SVM.

---
# Étape 2 — Exploration des profils temporels

Les analyses prévues sont :

- vérification de la fréquence temporelle ;
- visualisation de profils individuels ;
- analyse des consommations moyennes journalières et saisonnières ;
- comparaison entre logements.

Objectif :
identifier les structures temporelles exploitables par un modèle récurrent.

---

# Étape 3 — Préparation pour le modèle RNN

Les séries temporelles seront transformées en séquences d'apprentissage.

Exemple :

Entrée :

\[
X_t = [y_{t-n}, ..., y_{t-2}, y_{t-1}]
\]

où :
- \(y_t\) représente la consommation électrique à l'instant \(t\) ;
- \(n\) représente la longueur de la fenêtre temporelle.

Sortie :

\[
y_{t+1}
\]

ou un horizon futur plus long selon l'objectif choisi.

---

# Perspectives de modélisation

Deux architectures seront étudiées :

## 1. RNN simple

Une première approche baseline :


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'



import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

ELEC_COLS = [
    'out.electricity.total.energy_consumption..kwh',
    'out.natural_gas.total.energy_consumption..kwh',
    'out.fuel_oil.total.energy_consumption..kwh',
    'out.propane.total.energy_consumption..kwh'
]

Y_energy = pd.read_parquet(
    DATA_RAW / 'upgrade0.parquet',
    columns=ELEC_COLS
)

print(Y_energy.shape)
Y_energy.head()

# Étape 1 — Sélection du sous-ensemble étudié

Afin d'obtenir un ensemble homogène, on conserve uniquement les logements dont la consommation énergétique provient exclusivement de l'électricité.

Le filtrage est réalisé à partir des sorties `out.*.energy_consumption` :

- consommation électrique totale positive ;
- absence de consommation gaz ;
- absence de consommation fioul ;
- absence de consommation propane.

Cette sélection permet d'étudier un parc résidentiel homogène avant l'apprentissage temporel.



In [ ]:
# Charger uniquement les colonnes nécessaires
energy_cols = [
    'out.electricity.total.energy_consumption..kwh',
    'out.natural_gas.total.energy_consumption..kwh',
    'out.fuel_oil.total.energy_consumption..kwh',
    'out.propane.total.energy_consumption..kwh'
]

energy = pd.read_parquet(
    DATA_RAW / 'upgrade0.parquet',
    columns=energy_cols
)

# Recherche d'un logement uniquement électrique
mask_electric = (
    (energy['out.electricity.total.energy_consumption..kwh'] > 0) &
    (energy['out.natural_gas.total.energy_consumption..kwh'] == 0) &
    (energy['out.fuel_oil.total.energy_consumption..kwh'] == 0) &
    (energy['out.propane.total.energy_consumption..kwh'] == 0)
)

# Premier logement trouvé
bldg_idx = energy[mask_electric].index[101]

print("Index du bâtiment :", bldg_idx)